# Cross-family review gate (S5 Gold gate)

Runs the prepared 56-record package on a **different model family** and compares verdicts with the resolved candidate baseline. No training may start until this gate passes.

In [ ]:
# Havyn Stage 5 - environment setup and verification gate.
# Every notebook runs this first: no notebook trains or generates data on an
# unverified environment.
import sys, shutil, os
if not shutil.which("git"):
    raise RuntimeError("git missing")
!git clone --depth 1 https://github.com/HamzaNinjaCoder1/MCP-Packet-Tracer.git /content/MCP-Packet-Tracer 2>/dev/null || git -C /content/MCP-Packet-Tracer pull --ff-only
import sys
sys.path.insert(0, "/content/MCP-Packet-Tracer/havyn-stage5/scripts")
os.chdir("/content/MCP-Packet-Tracer")
from colab_setup import setup, save_report
env = setup(run_tests=True, use_drive=True)
assert not env.errors, f"environment verification failed: {env.errors}"
assert env.tests_passed, "offline suite failed - STOP, do not continue"
from env_report import build_report, recommend_config
report = build_report(repo="/content/MCP-Packet-Tracer", dataset_version="v0")
import json, pathlib
out_dir = pathlib.Path(env.drive_dir or "/content/havyn-artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "env-report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(recommend_config(report.get("gpu", {})), indent=2))
print(json.dumps(report, indent=2)[:2000])


In [ ]:
import os
os.environ["HAVYN_DRIVE"] = env.drive_dir or "/content/havyn-artifacts"
os.environ["HAVYN_REPO"] = "/content/MCP-Packet-Tracer"
print("HAVYN_DRIVE =", os.environ["HAVYN_DRIVE"])

Offline precheck first. It verifies the package SHA, language labels, refusal shape, duplicates, secrets and re-runs every record through the engine. It needs no GPU and writes `review/cross-family-precheck.json`.

In [ ]:
!python havyn-stage5/scripts/cross_family_review.py --precheck-only

### USER ACTION REQUIRED

1. Use a GPU runtime (>=24 GB for Mistral-Small-24B at 4-bit, or >=8 GB for a 4-bit 7B Mistral).
2. Add `REVIEW_MODEL_ID` to Colab Secrets, e.g. `mistralai/Mistral-Small-24B-Instruct-2501`. It must be a different family from the teacher (Qwen3) and the fine-tune base.
3. Set `DRY_RUN=true` to validate without loading the model.

The next cell is blocked unless the preflight passes.

In [ ]:
from colab_ux import gate, dry_run, warn_expensive
import os

assert os.environ.get("REVIEW_MODEL_ID"), (
    "set REVIEW_MODEL_ID in Colab Secrets (Mistral family, not Qwen)")

gate("cross_family", require_hf=True)
if dry_run():
    print("[DRY_RUN] reviewer model not loaded")
else:
    warn_expensive("CROSS-FAMILY REVIEW",
                   "loads a 24B-class model and reviews 56 records")
    !python havyn-stage5/scripts/cross_family_review.py --model "$REVIEW_MODEL_ID" --load-in-4bit

### Reading the result

The script writes `review/cross-family-reviews/local-transformers-<model>-<date>.json`. A run is usable only if all 56 records have a verdict. Any REJECT blocks the freeze gate; any REVIEW needs an owner decision. A reviewer ACCEPT is **not** owner approval.

In [ ]:
import glob, json
files = sorted(glob.glob("havyn-stage5/prompt-authoring/review/cross-family-reviews/*.json"))
if not files:
    print("no review file yet")
else:
    data = json.load(open(files[-1], encoding="utf-8"))
    print("model:", data["model"], "family:", data["family"])
    print("input SHA:", data["input_sha256"])
    print(json.dumps(data["summary"], indent=2))
    print("disagreements:", data["summary"]["disagreements"])